# Random Forests and Ensemble Methods

**Dataset:** Wine (sklearn built-in)  
**Task:** Multiclass classification — 3 wine cultivars

---

## Overview

**Ensemble methods** combine multiple weak learners to create a stronger predictor. The core intuition: many independent mediocre models that each make different errors tend to average out to something better than any individual model.

### Bagging (Bootstrap Aggregating)

1. Draw $B$ bootstrap samples (with replacement) from training data
2. Train a separate model on each bootstrap sample
3. Aggregate predictions by majority vote (classification) or averaging (regression)

**Variance reduction:** Because each model sees different data, their errors are less correlated, and averaging them reduces variance.

### Random Forests = Bagging + Feature Randomness

Random forests extend bagging by also randomly selecting a subset of features at each split. This **decorrelates** the trees further:

- At each node, consider only $m = \sqrt{p}$ randomly chosen features (classification)
- Prevents all trees from being dominated by the same strong features

### Gradient Boosting

Unlike bagging (parallel), **boosting** trains trees sequentially, where each new tree corrects the errors of the previous ensemble.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 2. Load and Explore the Wine Dataset

In [ ]:
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names
target_names  = data.target_names

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df['cultivar'] = y
df.describe()

In [ ]:
# Correlation heatmap of features
corr = pd.DataFrame(X, columns=feature_names).corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(feature_names)))
ax.set_yticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(feature_names, fontsize=9)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 4. Compare Models

We compare a single decision tree against bagging and random forests.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42
    ),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc  = model.score(X_test,  y_test)
    cv_scores = cross_val_score(model, X, y, cv=5)
    results[name] = {
        'Train Acc': train_acc,
        'Test Acc': test_acc,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std()
    }
    print(f"{name:20s} | Train: {train_acc:.3f} | Test: {test_acc:.3f} | CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# Bar chart comparison
names = list(results.keys())
test_accs = [results[n]['Test Acc'] for n in names]
cv_means  = [results[n]['CV Mean']  for n in names]
cv_stds   = [results[n]['CV Std']   for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, test_accs, width, label='Test Accuracy', color='steelblue', alpha=0.8)
ax.bar(x + width/2, cv_means,  width, label='CV Accuracy (5-fold)',
       color='tomato', alpha=0.8, yerr=cv_stds, capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison: Single Tree vs. Ensembles')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Effect of Number of Trees

In [ ]:
n_trees_range = [1, 5, 10, 25, 50, 100, 150, 200]
rf_train, rf_test = [], []

for n in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    rf_train.append(rf.score(X_train, y_train))
    rf_test.append(rf.score(X_test, y_test))

plt.figure(figsize=(9, 5))
plt.plot(n_trees_range, rf_train, 'o-', color='steelblue', label='Train accuracy')
plt.plot(n_trees_range, rf_test,  's--', color='tomato',    label='Test accuracy')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest: Effect of Number of Trees')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
rf = models['Random Forest']
importances = rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[sorted_idx],
        yerr=std[sorted_idx], color='steelblue', alpha=0.8, capsize=3)
plt.xticks(range(len(importances)), np.array(feature_names)[sorted_idx],
           rotation=45, ha='right', fontsize=9)
plt.ylabel('Mean Decrease in Impurity')
plt.title('Random Forest Feature Importance (with std dev)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary

| Model | Test Accuracy | Notes |
|-------|--------------|-------|
| Decision Tree | ~90% | High variance, can overfit |
| Bagging | ~95–97% | Reduces variance by averaging |
| Random Forest | ~97–100% | Adds feature randomness for decorrelation |
| Gradient Boosting | ~97–100% | Reduces bias sequentially |

### Key Takeaways

- **Bagging** reduces variance by training trees on different bootstrap samples in parallel.
- **Random forests** further decorrelate trees by considering only a random subset of features at each split.
- Accuracy **plateaus** after ~50–100 trees — more trees help but with diminishing returns.
- Feature importance scores from random forests are a useful **feature selection** tool.
- **Gradient boosting** often achieves the best accuracy but is slower to train and more prone to overfitting than random forests.